# ASR + Diarisation — Speaker-Attributed Transcript and Evaluation

Transcribes and diarises `how_irans_flagging_economy_inflamed_its_protests.mp3` using `mlx-community/whisper-large-v3-turbo` and `mlx-community/diar_sortformer_4spk-v1-fp32`, aligns ASR segments with diarisation speaker labels by maximum time overlap, then evaluates attribution quality against a reference transcript using `mlx-community/Qwen2.5-1.5B-Instruct-4bit` on Apple Silicon.

> **Why Whisper instead of Qwen3-ASR?** Qwen3-ASR timestamps require `Qwen3-ForcedAligner-0.6B`, which has no MLX port and only runs on CUDA. Whisper natively returns per-segment `(start, end, text)` timestamps, enabling direct time-overlap alignment with the diarisation output.

In [4]:
from pathlib import Path

import mlx.core as mx
from mlx_audio.stt.generate import generate_transcription
from mlx_audio.stt.utils import load_model as load_asr
from mlx_audio.vad import load as load_diar
from mlx_lm import generate, load
from transformers import WhisperProcessor

# Whisper returns native per-segment timestamps — required for time-overlap alignment
ASR_MODEL = "mlx-community/whisper-large-v3-turbo"
DIAR_MODEL = "mlx-community/diar_sortformer_4spk-v1-fp32"
EVAL_MODEL = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

AUDIO = Path("mlx-audio-test/original/audio/how_irans_flagging_economy_inflamed_its_protests.mp3")
TRANSCRIPT_DIR = Path("mlx-audio-test/generated/transcript")
DIARISATION_DIR = Path("mlx-audio-test/generated/diarisation")
ATTRIBUTED_DIR = Path("mlx-audio-test/generated/attributed")
COMPARISON_DIR = Path("mlx-audio-test/generated/comparison")
REFERENCE_TRANSCRIPT = Path("mlx-audio-test/original/transcript") / AUDIO.with_suffix(".md").name

## Transcription

In [5]:
asr_model = load_asr(ASR_MODEL)

# The mlx-community cache only stores weights + config (no tokenizer/processor files).
# Load the processor from the original OpenAI repo so get_tokenizer() works.
asr_model._processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [6]:
asr_result = generate_transcription(model=asr_model, audio=str(AUDIO))

100%|██████████| 51646/51646 [08:10<00:00, 105.39frames/s]


In [7]:
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
transcript_file = TRANSCRIPT_DIR / AUDIO.with_suffix(".md").name
transcript_file.write_text(asr_result.text)

del asr_model
mx.metal.clear_cache()
transcript_file

mx.metal.clear_cache is deprecated and will be removed in a future version. Use mx.clear_cache instead.


PosixPath('mlx-audio-test/generated/transcript/how_irans_flagging_economy_inflamed_its_protests.md')

## Diarisation

In [8]:
diar_model = load_diar(DIAR_MODEL)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
result = diar_model.generate(str(AUDIO), threshold=0.5, verbose=True)

Audio: 511.65s
Trimmed 0.69s leading silence
Features: (1, 80, 51168)
Found 1695 segments with 4 speakers
Processing time: 1549.04s


In [10]:
DIARISATION_DIR.mkdir(parents=True, exist_ok=True)

# Build a markdown table of all segments
lines = ["| Speaker | Start (s) | End (s) | Duration (s) |", "| --- | --- | --- | --- |"]
for seg in result.segments:
    duration = seg.end - seg.start
    lines.append(f"| Speaker {seg.speaker} | {seg.start:.2f} | {seg.end:.2f} | {duration:.2f} |")

diar_text = "\n".join(lines)
diar_file = DIARISATION_DIR / AUDIO.with_suffix(".md").name
diar_file.write_text(f"# Diarisation Output\n\n{diar_text}\n")
diar_file

PosixPath('mlx-audio-test/generated/diarisation/how_irans_flagging_economy_inflamed_its_protests.md')

In [11]:
del diar_model
mx.metal.clear_cache()

## Speaker Attribution

For each Whisper segment `(start, end, text)`, find the diarisation segment with the greatest time overlap and assign that speaker label. Consecutive segments from the same speaker are then merged into turns.

In [12]:
def _overlap(a_start: float, a_end: float, b_start: float, b_end: float) -> float:
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def assign_speakers(asr_segments, diar_segments) -> list[dict]:
    """Assign a speaker label to each Whisper ASR segment by maximum time overlap.

    Whisper segments are dicts with keys 'start', 'end', 'text'.
    Diarisation segments are objects with .start, .end, .speaker attributes.
    """
    attributed = []
    for seg in asr_segments:
        best_speaker = "Unknown"
        best_overlap = 0.0
        for diar in diar_segments:
            ov = _overlap(seg["start"], seg["end"], diar.start, diar.end)
            if ov > best_overlap:
                best_overlap = ov
                best_speaker = f"Speaker {diar.speaker}"
        attributed.append({"speaker": best_speaker, "text": seg["text"].strip()})
    return attributed


def group_turns(attributed: list[dict]) -> list[dict]:
    """Merge consecutive segments from the same speaker into turns."""
    turns: list[dict] = []
    for item in attributed:
        if turns and turns[-1]["speaker"] == item["speaker"]:
            turns[-1]["text"] += " " + item["text"]
        else:
            turns.append({"speaker": item["speaker"], "text": item["text"]})
    return turns

In [13]:
turns = group_turns(assign_speakers(asr_result.segments, result.segments))

# Preview first 5 turns
for turn in turns[:5]:
    print(f"**{turn['speaker']}**: {turn['text'][:120]}...")

**Speaker 0**: NPR....
**Speaker 1**: As we speak, U.S. Navy troops are in a state of waiting. They're telling their husbands, wives, and children that they'r...
**Speaker 0**: Iran's leadership is in a vulnerable position. This comes off the back of massive countrywide protests that swept the na...
**Speaker 1**: They've mostly been quelled for now because of a bloody crackdown by the Iranian government. So far, activists say that ...
**Speaker 0**: And I'm Darian Woods. Today on the show, the economic roots of Iran's protests. We speak with a small business owner in ...


In [14]:
ATTRIBUTED_DIR.mkdir(parents=True, exist_ok=True)
attributed_text = "\n\n".join(f"**{t['speaker']}**: {t['text']}" for t in turns)
attributed_file = ATTRIBUTED_DIR / AUDIO.with_suffix(".md").name
attributed_file.write_text(f"# Speaker-Attributed Transcript\n\n{attributed_text}\n")
attributed_file

PosixPath('mlx-audio-test/generated/attributed/how_irans_flagging_economy_inflamed_its_protests.md')

## Speaker-Attributed Transcript Evaluation

Uses `Qwen2.5-1.5B-Instruct-4bit` to evaluate the speaker-attributed transcript against the reference on four dimensions:
1. Speaker count — does the number of distinct speakers match the reference?
2. Speaker consistency — is each speaker label used consistently across turns?
3. Turn-taking accuracy — do speaker boundaries align with the reference?
4. Text–speaker alignment — is the transcribed text attributed to the correct speaker?

In [15]:
attributed_text = attributed_file.read_text()
reference_text = REFERENCE_TRANSCRIPT.read_text()

In [16]:
llm, tokenizer = load(EVAL_MODEL)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [17]:
prompt_text = f"""You are evaluating a speaker-attributed transcript produced by aligning ASR output with speaker diarisation. The goal is to assess whether the transcribed text is correctly attributed to the right speakers.

## Reference transcript
{reference_text}

## Speaker-attributed transcript
{attributed_text}

Evaluate the speaker-attributed transcript on the following four dimensions. For each, give a rating (Excellent / Good / Fair / Poor) and a brief explanation with specific examples where relevant.

### 1. Speaker count
Does the number of distinct speakers in the attributed transcript match the reference? Note any over- or under-clustering.

### 2. Speaker consistency
Is each speaker label used consistently throughout? Note any turns where the same speaker from the reference appears under different labels.

### 3. Turn-taking accuracy
Do the speaker turn boundaries align with the speaker changes implied by the reference? Note any missed transitions or spurious splits.

### 4. Text–speaker alignment
Is the transcribed text attributed to the correct speaker? Note any segments where the content is clearly from the wrong speaker.

### Summary
One paragraph overall assessment of the speaker attribution quality.
"""

messages = [{"role": "user", "content": prompt_text}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
evaluation = generate(llm, tokenizer, prompt=prompt, max_tokens=1500, verbose=False)

mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


In [18]:
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
evaluation_file = COMPARISON_DIR / f"attributed_evaluation_{AUDIO.stem}.md"
evaluation_file.write_text(f"# Speaker Attribution Evaluation\n\n{evaluation}\n")

del llm, tokenizer
mx.metal.clear_cache()